# [3장 1강] - Attention 필요성과 Query/Key/Value (1)

<aside>
🎯

**실습 목표**

- 입력 표현에서 Query, Key, Value를 직접 계산합니다.
- Query와 Key의 내적으로 token 간 관련도를 구합니다.
- Attention score를 token 이름과 연결해 해석 가능한 리포트로 만듭니다.
</aside>

<aside>
💬

**튜터 한마디**

Q, K, V는 서로 다른 데이터가 아니라 같은 입력을 서로 다른 관점으로 투영한 결과입니다. Shape와 역할을 함께 확인하세요.

</aside>

---

## 핵심 실습. Q·K·V Projection 함수 작성

### 시작 코드

```python
import torch

X = torch.tensor([
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0],
])

W_q = torch.tensor([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])
W_k = torch.tensor([[0.5, 0.0], [0.0, 0.5], [1.0, 1.0]])
W_v = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, -1.0]])
```

### 수행해야 할 작업

1. `project_qkv(X, W_q, W_k, W_v)` 함수를 작성하세요.
2. 입력 차원과 각 weight의 첫 차원이 맞는지 검사하세요.
3. Q, K, V를 딕셔너리로 반환하세요.
4. 각 Tensor의 shape를 출력하세요.
    
  **해설**
    
   세 projection은 token 수를 바꾸지 않고 마지막 표현 차원만 바꿉니다. Q는 현재 token이 찾는 정보, K는 자신이 어떤 정보와 연결되는지 나타내는 기준, V는 최종 context에 섞일 내용으로 이해할 수 있습니다.

In [1]:
import torch

X = torch.tensor([[1., 0., 1.], [0., 1., 1.], [1., 1., 0.]])
W_q = torch.tensor([[1., 0.], [0., 1.], [0.5, 0.5]])
W_k = torch.tensor([[0.5, 0.], [0., 0.5], [1., 1.]])
W_v = torch.tensor([[1., 0.], [0., 1.], [1., -1.]])


def project_qkv(X, W_q, W_k, W_v):
    input_dim = X.size(-1)
    for name, weight in {"W_q": W_q, "W_k": W_k, "W_v": W_v}.items():
        if weight.size(0) != input_dim:
            raise ValueError(f"{name}의 입력 차원이 맞지 않습니다.")

    # 같은 입력 X를 서로 다른 학습 weight에 통과시킵니다.
    return {"Q": X @ W_q, "K": X @ W_k, "V": X @ W_v}


qkv = project_qkv(X, W_q, W_k, W_v)
for name, tensor in qkv.items():
    print(name, tensor, tuple(tensor.shape))

assert all(tensor.shape == (3, 2) for tensor in qkv.values())

Q tensor([[1.5000, 0.5000],
        [0.5000, 1.5000],
        [1.0000, 1.0000]]) (3, 2)
K tensor([[1.5000, 1.0000],
        [1.0000, 1.5000],
        [0.5000, 0.5000]]) (3, 2)
V tensor([[ 2., -1.],
        [ 1.,  0.],
        [ 1.,  1.]]) (3, 2)


## 핵심 보조 실습. Query별 가장 관련 있는 Key 찾기

### 시작 코드

```python
Q = torch.tensor([[1.0, 0.0], [0.5, 1.0], [0.0, 1.0]])
K = torch.tensor([[1.0, 0.0], [0.0, 1.0], [0.8, 0.2]])
tokens = ["모델", "보안", "배포"]
```

### 수행해야 할 작업

1. `find_top_keys(Q, K, tokens)` 함수를 작성하세요.
2. `Q @ K.T`로 score 행렬을 만드세요.
3. 각 Query에서 자기 자신을 제외하고 가장 높은 Key를 찾으세요.
4. Query token, 선택된 Key token, score를 반환하세요.

    
  **해설**
    
  Dot product가 크다는 것은 현재 projection 공간에서 방향이 잘 맞는다는 뜻입니다. 아직 softmax를 적용하지 않았으므로 이 값은 확률이 아니라 raw compatibility score입니다.

In [2]:
import torch

Q = torch.tensor([[1., 0.], [0.5, 1.], [0., 1.]])
K = torch.tensor([[1., 0.], [0., 1.], [0.8, 0.2]])
tokens = ["모델", "보안", "배포"]


def find_top_keys(Q, K, tokens):
    if Q.size(0) != len(tokens) or K.size(0) != len(tokens):
        raise ValueError("token 수와 Tensor의 sequence 차원이 다릅니다.")

    scores = Q @ K.T
    masked_scores = scores.clone()
    masked_scores.fill_diagonal_(float("-inf"))
    top_indices = masked_scores.argmax(dim=-1)

    links = []
    for query_index, key_index in enumerate(top_indices.tolist()):
        links.append({
            "query": tokens[query_index],
            "key": tokens[key_index],
            "score": float(scores[query_index, key_index]),
        })
    return scores, links


scores, links = find_top_keys(Q, K, tokens)
print(scores)
print(links)

assert links[0]["key"] == "배포"
assert links[2]["key"] == "보안"

tensor([[1.0000, 0.0000, 0.8000],
        [0.5000, 1.0000, 0.6000],
        [0.0000, 1.0000, 0.2000]])
[{'query': '모델', 'key': '배포', 'score': 0.800000011920929}, {'query': '보안', 'key': '배포', 'score': 0.6000000238418579}, {'query': '배포', 'key': '보안', 'score': 1.0}]


## 참고·심화 실습. Token 관계 리포트 생성

### 시작 코드

```python
score_matrix = torch.tensor([
    [2.0, 0.2, 1.1],
    [0.3, 2.2, 0.7],
    [1.4, 0.5, 1.8],
])
tokens = ["private", "llm", "serving"]
```

### 수행해야 할 작업

1. 각 행에 softmax를 적용하세요.
2. 자기 자신을 제외한 상위 관계와 weight를 찾으세요.
3. weight가 0.3 이상인 관계만 남기는 `build_relation_report` 함수를 작성하세요.
4. 각 행의 weight 합이 1인지 검증하세요.
    ```
    
  **해설**
    
  Attention weight는 모델 내부 연결을 관찰하는 단서이지만, 그 자체를 인과적 설명으로 단정하면 안 됩니다. 여러 head와 layer, 입력 변화에 따른 일관성을 함께 확인하세요.

In [ ]:
import torch

score_matrix = torch.tensor([[2.0, 0.2, 1.1], [0.3, 2.2, 0.7], [1.4, 0.5, 1.8]])
tokens = ["private", "llm", "serving"]


def build_relation_report(scores, tokens, threshold=0.3):
    weights = torch.softmax(scores, dim=-1)
    reports = []

    for i, query in enumerate(tokens):
        candidates = [
            (j, float(weights[i, j]))
            for j in range(len(tokens))
            if j != i
        ]
        key_index, weight = max(candidates, key=lambda item: item[1])
        if weight >= threshold:
            reports.append({"query": query, "key": tokens[key_index], "weight": round(weight, 4)})

    return weights, reports


weights, reports = build_relation_report(score_matrix, tokens)
print(weights)
print(reports)
assert torch.allclose(weights.sum(dim=-1), torch.ones(3))